# Batch analysis: C++ vs Python parity

Requires `../RingDownAnalysis/data` (CSV/MAT fixtures) or set `RINGDOWN_EXAMPLES_DATA`.

This workflow analyzes all top-level CSV/MAT files in the data directory and writes reference-notebook-style plots for both Python and C++ outputs. The notebook cell prints Python and C++ compute times before running the comparison command, which writes plots before returning nonzero when parity mismatches remain.

The C++ step uses the Release binary and enables progress output. By default, the C++ batch report omits raw waveform arrays for speed; add `--notebook-report` to the C++ command when waveform-heavy time-series plots are needed.

```bash
.venv/bin/python examples/python/export_batch_reference.py --n-jobs -1
cmake --build build/release
build/release/examples/batch_analysis_example --workers 2 --progress
.venv/bin/python examples/python/compare_batch_analysis.py \
  --py-report results/examples/batch_analysis_py/batch_report.json \
  --cpp-report results/examples/batch_analysis_cpp/batch_report.json \
  --plot results/examples/batch_analysis_cpp/f_nls_overlay.png \
  --plot-dir results/examples/batch_analysis_plots
```


In [ ]:
import subprocess
import time
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "examples/python/export_batch_reference.py").exists():
    ROOT = ROOT.parent

PYTHON = ROOT / ".venv/bin/python"
RELEASE_BUILD = ROOT / "build/release"
CPP_BATCH = RELEASE_BUILD / "examples/batch_analysis_example"
# Python and C++ defaults resolve measurement data from ../RingDownAnalysis/data.

if not RELEASE_BUILD.exists():
    raise FileNotFoundError(
        "Release build directory is missing; configure it with: "
        "cmake -S . -B build/release -DCMAKE_BUILD_TYPE=Release"
    )


def run_timed(label, args):
    start = time.perf_counter()
    try:
        subprocess.check_call(args, cwd=ROOT)
    finally:
        elapsed = time.perf_counter() - start
        print(f"{label} compute time: {elapsed:.3f} s")


run_timed(
    "Python",
    [str(PYTHON), str(ROOT / "examples/python/export_batch_reference.py"), "--n-jobs", "-1"],
)
subprocess.check_call(["cmake", "--build", str(RELEASE_BUILD)], cwd=ROOT)
if not CPP_BATCH.exists():
    raise FileNotFoundError(f"Release C++ batch binary was not produced: {CPP_BATCH}")
run_timed(
    "C++",
    [str(CPP_BATCH), "--workers", "2", "--progress"],
)
comparison = subprocess.run(
    [
        str(PYTHON),
        str(ROOT / "examples/python/compare_batch_analysis.py"),
        "--py-report",
        str(ROOT / "results/examples/batch_analysis_py/batch_report.json"),
        "--cpp-report",
        str(ROOT / "results/examples/batch_analysis_cpp/batch_report.json"),
        "--plot",
        str(ROOT / "results/examples/batch_analysis_cpp/f_nls_overlay.png"),
        "--plot-dir",
        str(ROOT / "results/examples/batch_analysis_plots"),
    ],
    cwd=ROOT,
    check=False,
)
if comparison.returncode != 0:
    print(f"Comparison exited with status {comparison.returncode}; inspect mismatch output above.")
print("Done.")
